<a href="https://colab.research.google.com/github/pramodkumarw/Github-Colab/blob/main/%20Iterative_Workflows.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#Langgraph

In [ ]:
# STEP 1: SET UP THE ENVIRONMENT

# STEP 1.1 INSTALL THE REQUIRED PACKAGES
!pip install langchain_community
!pip install -U duckduckgo-search
!pip install langchain-groq

In [6]:
from langchain_groq import ChatGroq
from dotenv import load_dotenv
import os
# from rich import print
from google.colab import userdata

GROQ_API_KEY=userdata.get("GROQ_API_KEY")
llm=ChatGroq(model="openai/gpt-oss-120b", groq_api_key=GROQ_API_KEY)


In [ ]:
!pip install langchain
!pip install langgrah


In [34]:
from pydantic import BaseModel
from typing import Literal,operator,Annotated,TypedDict
from langgraph.graph import StateGraph,START,END
from langchain.messages import HumanMessage, SystemMessage

class TweetState(TypedDict):
  topic:str
  tweet:str
  evaluation:Literal["approved","need_improvement"]
  feedback:str
  iteration:int
  max_iteration:int

  tweet_history: Annotated[list[str], operator.add]
  feedback_history: Annotated[list[str], operator.add]

In [35]:
def generate_tweet(state:TweetState):
  messages=[
      SystemMessage(content="you are a funny and cleaver tweeter infulencer"),
     HumanMessage(content=f"""
          Write a short, original, and hilarious tweet on the topic: "{state['topic']}".

          Rules:
          - Do NOT use question-answer format.
          - Max 280 characters.
          - Use observational humor, irony, sarcasm, or cultural references.
          - Think in meme logic, punchlines, or relatable takes.
          - Use simple, day to day english
          """)
  ]
  response=llm.invoke(messages).content
  return {'tweet': response, 'tweet_history': [response]}

In [42]:
from pydantic import BaseModel, Field

class TweetEvaluation(BaseModel):
    evaluation: Literal["approved", "needs_improvement"] = Field(..., description="Final evaluation result.")
    feedback: str = Field(..., description="feedback for the tweet.")

structured_evaluator_llm = llm.with_structured_output(TweetEvaluation)

def evaluate_tweet(state:TweetState):
  messages = [
    SystemMessage(content="You are a ruthless, no-laugh-given Twitter critic. You evaluate tweets based on humor, originality, virality, and tweet format."),
    HumanMessage(content=f"""
          Evaluate the following tweet:

          Tweet: "{state['tweet']}"

          Use the criteria below to evaluate the tweet:

          1. Originality – Is this fresh, or have you seen it a hundred times before?
          2. Humor – Did it genuinely make you smile, laugh, or chuckle?
          3. Punchiness – Is it short, sharp, and scroll-stopping?
          4. Virality Potential – Would people retweet or share it?
          5. Format – Is it a well-formed tweet (not a setup-punchline joke, not a Q&A joke, and under 280 characters)?

          Auto-reject if:
          - It's written in question-answer format (e.g., "Why did..." or "What happens when...")
          - It exceeds 280 characters
          - It reads like a traditional setup-punchline joke
          - Dont end with generic, throwaway, or deflating lines that weaken the humor (e.g., “Masterpieces of the auntie-uncle universe” or vague summaries)

          ### Respond ONLY in structured format:
          - evaluation: "approved" or "needs_improvement"
          - feedback: One paragraph explaining the strengths and weaknesses
          """)
]
  response=structured_evaluator_llm.invoke(messages)
  return {'evaluation':response.evaluation, 'feedback': response.feedback, 'feedback_history': [response.feedback]}

In [43]:
def optimize_tweet(state: TweetState):

    messages = [
        SystemMessage(content="You punch up tweets for virality and humor based on given feedback."),
        HumanMessage(content=f"""
            Improve the tweet based on this feedback:
            "{state['feedback']}"

            Topic: "{state['topic']}"
            Original Tweet:
            {state['tweet']}

            Re-write it as a short, viral-worthy tweet. Avoid Q&A style and stay under 280 characters.
            """)
        ]

    response = llm.invoke(messages).content
    iteration = state['iteration'] + 1

    return {'tweet': response, 'iteration': iteration, 'tweet_history': [response]}

In [44]:
def route_evaluation(state: TweetState):

    if state['evaluation'] == 'approved' or state['iteration'] >= state['max_iteration']:
        return 'approved'
    else:
        return 'needs_improvement'

In [45]:
graph = StateGraph(TweetState)

graph.add_node('generate', generate_tweet)
graph.add_node('evaluate', evaluate_tweet)
graph.add_node('optimize', optimize_tweet)

graph.add_edge(START, 'generate')
graph.add_edge('generate', 'evaluate')

graph.add_conditional_edges('evaluate', route_evaluation, {'approved': END, 'needs_improvement': 'optimize'})
graph.add_edge('optimize', 'evaluate')

workflow = graph.compile()


In [52]:
initial_state = {
    "topic": "srhberhb",
    "iteration": 1,
    "max_iteration": 5
}
result = workflow.invoke(initial_state)
result

{'topic': 'srhberhb',
 'tweet': 'srhberhb – the only word my brain spells after a coffee and a deadline. It’s basically “I need sleep” in a language only my keyboard understands. #MondayMood',
 'evaluation': 'approved',
 'feedback': 'The tweet feels fresh with its invented "srhberhb" as a tongue‑in‑cheek code for exhaustion, which gives it a nice original spin. The humor lands by visualizing a keyboard‑only language, earning a modest chuckle. It’s concise enough to scroll‑stop while still delivering the punch, and the #MondayMood tag boosts shareability among coffee‑fuelled professionals. Overall it meets all format rules and has decent virality potential.',
 'iteration': 1,
 'max_iteration': 5,
 'tweet_history': ['srhberhb – the only word my brain spells after a coffee and a deadline. It’s basically “I need sleep” in a language only my keyboard understands. #MondayMood'],
 'feedback_history': ['The tweet feels fresh with its invented "srhberhb" as a tongue‑in‑cheek code for exhaustion

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [50]:
for tweet in result['tweet_history']:
    print(tweet)


Just typed “srhberhb” into my phone and now it thinks I’m speaking an ancient dialect. Guess I’m fluent in “keyboard smash”—the only language that gets auto‑corrected to “sorry, I didn’t get that.” 🤖💬 #TechFails
Accidentally typed “srhberhb” and my phone launched a full‑blown debate in an ancient dialect. Now it refuses to autocorrect anything else. 🤖🗣️ #KeyboardChaos
